In [1]:
import torch 
import json 
import os
import cv2
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader 
from PIL import Image, ImageFile 
import torchvision.transforms as T

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor



In [2]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
IMAGE_DIR = "../data/train_images"
ANNOTATION_PATH = "../data/train_annotations.json"
with open(ANNOTATION_PATH, "r") as f:
    train_annotations = json.load(f)
    
print("Total annotations:", len(train_annotations))

Total annotations: 9580


In [5]:
def sanitize_boxes(boxes, labels):
    x1, y1, x2, y2 = boxes.unbind(1)
    keep = (x2 > x1) & (y2 > y1)
    return boxes[keep], labels[keep]

In [6]:
class DamageDataset(Dataset):
    def __init__(self, images_dir, annotations, transforms=None):
        self.images_dir = images_dir
        self.annotations = annotations
        self.transforms = transforms

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_path = os.path.join(self.images_dir, ann["image_name"])

        try:
            # --- READ IMAGE WITH CV2 ---
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        except:
            return None

        boxes = torch.tensor(ann["boxes"], dtype=torch.float32)
        labels = torch.tensor(ann["labels"], dtype=torch.int64)

        boxes, labels = sanitize_boxes(boxes, labels)
        if boxes.shape[0] == 0:
            return None

        # ---- AUGMENTATION: HORIZONTAL FLIP ----
        if np.random.rand() > 0.5:
            image = np.fliplr(image).copy()
            h, w, _ = image.shape

            if boxes.shape[0] > 0:
                boxes[:, [0, 2]] = w - boxes[:, [2, 0]]

        # convert image to tensor
        image = Image.fromarray(image)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 1] - boxes[:, 3]).abs(),
            "iscrowd": torch.zeros((boxes.shape[0],), dtype=torch.int64)
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target


full_dataset = DamageDataset(
    IMAGE_DIR,
    train_annotations,
    T.Compose([T.ToTensor()])
)

dataset = torch.utils.data.Subset(full_dataset, range(6000))
print("Training samples:", len(dataset))


Training samples: 6000


In [7]:
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    return tuple(zip(*batch))

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory = True
           
)


In [8]:
NUM_CLASSES = 5

model = fasterrcnn_resnet50_fpn(weights="DEFAULT", min_size=600, max_size=800)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features, NUM_CLASSES
)

model = model.to(device)
model.train()

print("Model ready ✔")


Model ready ✔


In [9]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.9,
    weight_decay=0.0005
)


In [10]:
torch.cuda.empty_cache()


In [12]:
num_epochs = 20          # upper bound
best_loss = float("inf")
patience = 5             # stop if no improvement
patience_counter = 0

for epoch in range(num_epochs):
    total_loss = 0.0
    model.train()

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}: Avg Loss = {avg_loss:.4f}")

    # 🔑 EARLY STOPPING
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
        print("✔ Best model saved")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print("🛑 Early stopping triggered")
        break


Epoch 1: Avg Loss = 0.4346
✔ Best model saved
Epoch 2: Avg Loss = 0.3965
✔ Best model saved
Epoch 3: Avg Loss = 0.3804
✔ Best model saved
Epoch 4: Avg Loss = 0.3656
✔ Best model saved
Epoch 5: Avg Loss = 0.3552
✔ Best model saved
Epoch 6: Avg Loss = 0.3458
✔ Best model saved
Epoch 7: Avg Loss = 0.3310
✔ Best model saved
Epoch 8: Avg Loss = 0.3170
✔ Best model saved
Epoch 9: Avg Loss = 0.3042
✔ Best model saved
Epoch 10: Avg Loss = 0.2866
✔ Best model saved
Epoch 11: Avg Loss = 0.2753
✔ Best model saved
Epoch 12: Avg Loss = 0.2604
✔ Best model saved
Epoch 13: Avg Loss = 0.2495
✔ Best model saved
Epoch 14: Avg Loss = 0.2373
✔ Best model saved
Epoch 15: Avg Loss = 0.2244
✔ Best model saved
Epoch 16: Avg Loss = 0.2159
✔ Best model saved
Epoch 17: Avg Loss = 0.2054
✔ Best model saved
Epoch 18: Avg Loss = 0.1966
✔ Best model saved
Epoch 19: Avg Loss = 0.1892
✔ Best model saved
Epoch 20: Avg Loss = 0.1858
✔ Best model saved


In [13]:
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_loss': best_loss
}, "checkpoint.pth")


In [11]:
checkpoint = torch.load("checkpoint.pth", map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1
best_loss = checkpoint['best_loss']

C:\Users\HP\AppData\Local\Temp\ipykernel_28088\542783267.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("checkpoint.pth", map_location=device)


In [13]:
print(start_epoch, best_loss)

20 0.1857890942469239


In [14]:
num_epochs = 20          # additional epochs
patience = 5

for epoch in range(start_epoch, start_epoch + num_epochs):

    total_loss = 0.0
    model.train()

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}: Avg Loss = {avg_loss:.4f}")

    # Early stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss': best_loss,
        }, "checkpoint.pth")

        print("✔ Best model saved")

    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print("🛑 Early stopping triggered")
        break

Epoch 21: Avg Loss = 0.1792
✔ Best model saved
Epoch 22: Avg Loss = 0.1718
✔ Best model saved
Epoch 23: Avg Loss = 0.1659
✔ Best model saved
Epoch 24: Avg Loss = 0.1632
✔ Best model saved
Epoch 25: Avg Loss = 0.1579
✔ Best model saved
Epoch 26: Avg Loss = 0.1553
✔ Best model saved
Epoch 27: Avg Loss = 0.1519
✔ Best model saved
Epoch 28: Avg Loss = 0.1479
✔ Best model saved
Epoch 29: Avg Loss = 0.1439
✔ Best model saved
Epoch 30: Avg Loss = 0.1442
No improvement (1/5)
Epoch 31: Avg Loss = 0.1427
✔ Best model saved
Epoch 32: Avg Loss = 0.1390
✔ Best model saved
Epoch 33: Avg Loss = 0.1390
✔ Best model saved
Epoch 34: Avg Loss = 0.1336
✔ Best model saved
Epoch 35: Avg Loss = 0.1335
✔ Best model saved
Epoch 36: Avg Loss = 0.1315
✔ Best model saved
Epoch 37: Avg Loss = 0.1278
✔ Best model saved
Epoch 38: Avg Loss = 0.1275
✔ Best model saved
Epoch 39: Avg Loss = 0.1307
No improvement (1/5)
Epoch 40: Avg Loss = 0.1285
No improvement (2/5)


In [15]:
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_loss': best_loss
}, "checkpoint.pth")


In [16]:
checkpoint = torch.load("checkpoint.pth", map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1
best_loss = checkpoint['best_loss']

C:\Users\HP\AppData\Local\Temp\ipykernel_28088\542783267.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("checkpoint.pth", map_location=device)


In [17]:
print(start_epoch, best_loss)

40 0.12751747638732194
